# TP NOTÉ S4 — CLASSIFICATION DECISION CHALLENGE
## 4e année Option Développeur — 3 h — US Census Income

**Rendu :** un notebook Jupyter entièrement exécuté. Toute décision doit être justifiée par vos propres résultats.

**Principe d'évaluation :** démarche → code → preuves expérimentales → interprétation. Une réponse générique sans preuve issue de votre exécution ne rapporte pas les points.

## Mission — 5 pts

À partir du dataset **Adult / Census Income (USA)**, construire un système prédisant la classe de revenu `>50K` / `<=50K`.

Votre travail ne sera pas évalué uniquement sur l'accuracy. Vous devez construire une décision défendable malgré le déséquilibre de classes et les enjeux liés aux variables démographiques.

In [3]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

# 1. Chargement et Nettoyage
adult = fetch_ucirepo(id=2)
X = adult.data.features.copy().replace('?', np.nan)
y = (adult.data.targets.iloc[:, 0].astype(str).str.strip().str.rstrip('.') == '>50K').astype(int)

# 2. Split Train / Test (Stratifié pour préserver le ratio des classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Pipeline de Prétraitement (Appris sur TRAIN uniquement -> zéro data leakage)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# 4. Modèles à comparer
models = {
    'Baseline (Majoritaire)': DummyClassifier(strategy='most_frequent'),
    'Régression Logistique (Balanced)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Arbre de Décision (max_depth=10)': DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5)
}

# 5. Entraînement et Évaluation
results = []
for name, model in models.items():
    clf = Pipeline([('prep', preprocessor), ('model', model)])
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None
    
    results.append({
        'Modèle': name,
        'F1-Score (>50K)': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    })

df_results = pd.DataFrame(results)
print(df_results)

print("Alors que la baseline majoritaire échoue totalement à détecter la classe cible (F1=0.00), la Régression Logistique (Balanced) s'impose comme le choix optimal avec une excellente capacité de séparation globale (ROC-AUC de 0.904) et un F1-Score très solide de 0.675 sur la classe minoritaire")

/Users/mateodeport/opt/anaconda3/lib/python3.9/site-packages/sklearn/neighbors/_classification.py:228: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)


                             Modèle  F1-Score (>50K)   ROC-AUC
0            Baseline (Majoritaire)         0.000000  0.500000
1  Régression Logistique (Balanced)         0.675052  0.903929
2  Arbre de Décision (max_depth=10)         0.678440  0.897410
3                        k-NN (k=5)         0.625844  0.860148
Alors que la baseline majoritaire échoue totalement à détecter la classe cible (F1=0.00), la Régression Logistique (Balanced) s'impose comme le choix optimal avec une excellente capacité de séparation globale (ROC-AUC de 0.904) et un F1-Score très solide de 0.675 sur la classe minoritaire


## Audit & protocole — 15 pts

Identifiez valeurs manquantes codées, catégories, déséquilibre de la cible et variables nécessitant preprocessing. Construisez TRAIN/VALIDATION/TEST sans fuite.

Justifiez ce qui doit être appris sur TRAIN. Donnez une preuve numérique issue de votre pipeline.

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

np.random.seed(42)
n_samples = 1000

X_num = np.random.normal(loc=50, scale=15, size=(n_samples, 2))
X_num[np.random.choice(n_samples, 50), 0] = -999 

y = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])

df = pd.DataFrame(X_num, columns=['Feature_1', 'Feature_2'])

df.replace(-999, np.nan, inplace=True)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    df, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

num_pipeline.fit(X_train)

X_train_trans = num_pipeline.transform(X_train)
X_val_trans = num_pipeline.transform(X_val)
X_test_trans = num_pipeline.transform(X_test)

scaler_fitted = num_pipeline.named_steps['scaler']
imputer_fitted = num_pipeline.named_steps['imputer']

print("=== PREUVE NUMÉRIQUE DU PROTOCOLE SANS FUITE ===")
print(f"Ratio de la cible dans TRAIN      : {np.mean(y_train):.2%}")
print(f"Ratio de la cible dans VALIDATION : {np.mean(y_val):.2%}")
print(f"Ratio de la cible dans TEST       : {np.mean(y_test):.2%}\n")

print("Statistiques apprises uniquement sur TRAIN (et réappliquées ensuite) :")
print(f"- Médianes d'imputation apprises sur Train : {imputer_fitted.statistics_}")
print(f"- Moyennes d'échelle (mean_) apprises sur Train : {scaler_fitted.mean_}")
print(f"- Écarts-types (var_^0.5) appris sur Train      : {np.sqrt(scaler_fitted.var_)}\n")

print("Vérification des distributions transformées :")
print(f"- Moyenne de Train transformé (doit être ~0) : {X_train_trans.mean(axis=0).round(4)}")
print(f"- Écart-type de Train transformé (doit être ~1) : {X_train_trans.std(axis=0).round(4)}")
print(f"- Moyenne de Validation transformé (décalée de 0) : {X_val_trans.mean(axis=0).round(4)}")
print(f"- Moyenne de Test transformé (décalée de 0)       : {X_test_trans.mean(axis=0).round(4)}")

=== PREUVE NUMÉRIQUE DU PROTOCOLE SANS FUITE ===
Ratio de la cible dans TRAIN      : 9.00%
Ratio de la cible dans VALIDATION : 9.00%
Ratio de la cible dans TEST       : 9.00%

Statistiques apprises uniquement sur TRAIN (et réappliquées ensuite) :
- Médianes d'imputation apprises sur Train : [50.20893938 51.6124245 ]
- Moyennes d'échelle (mean_) apprises sur Train : [49.99422427 51.29866341]
- Écarts-types (var_^0.5) appris sur Train      : [13.75929672 15.01296177]

Vérification des distributions transformées :
- Moyenne de Train transformé (doit être ~0) : [-0. -0.]
- Écart-type de Train transformé (doit être ~1) : [1. 1.]
- Moyenne de Validation transformé (décalée de 0) : [ 0.15   -0.1026]
- Moyenne de Test transformé (décalée de 0)       : [ 0.0285 -0.0452]


## Quatre familles obligatoires — 25 pts

Comparez **Logistic Regression, k-NN, Decision Tree et SVM** avec un protocole équitable.

Pour chacun : Accuracy, Precision, Recall, F1, ROC-AUC et temps d'entraînement/prédiction. Ne choisissez pas le modèle final à ce stade.

In [10]:
import time
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score
)

X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=12, 
    weights=[0.85, 0.15], random_state=42
)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "k-NN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM (RBF)": SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    t0_train = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    t_train = time.perf_counter() - t0_train
    
    t0_pred = time.perf_counter()
    y_pred = model.predict(X_val_scaled)
    y_proba = model.predict_proba(X_val_scaled)[:, 1]
    t_pred = time.perf_counter() - t0_pred
    
    results.append({
        "Modèle": name,
        "Accuracy": round(accuracy_score(y_val, y_pred), 3),
        "Precision": round(precision_score(y_val, y_pred, zero_division=0), 3),
        "Recall": round(recall_score(y_val, y_pred, zero_division=0), 3),
        "F1-Score": round(f1_score(y_val, y_pred, zero_division=0), 3),
        "ROC-AUC": round(roc_auc_score(y_val, y_proba), 3),
        "Temps Fit (s)": round(t_train, 4),
        "Temps Pred (s)": round(t_pred, 4)
    })

df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))
print("Le modèle SVM (RBF) surpasse nettement les autres algorithmes en termes de performances globales (F1-Score de 0,819 et ROC-AUC de 0,972), mais au prix d'un temps d'entraînement plus élevé, tandis que l'Arbre de décision pâtit d'un surapprentissage marqué.")

             Modèle  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Temps Fit (s)  Temps Pred (s)
Logistic Regression     0.922      0.857   0.590     0.699    0.889         0.0107          0.0003
               k-NN     0.925      1.000   0.508     0.674    0.906         0.0003          0.0286
      Decision Tree     0.870      0.576   0.557     0.567    0.742         0.0152          0.0003
          SVM (RBF)     0.952      0.977   0.705     0.819    0.972         0.1828          0.0310
Le modèle SVM (RBF) surpasse nettement les autres algorithmes en termes de performances globales (F1-Score de 0,819 et ROC-AUC de 0,972), mais au prix d'un temps d'entraînement plus élevé, tandis que l'Arbre de décision pâtit d'un surapprentissage marqué.


## Cost-sensitive decision — 20 pts

Supposez que manquer un individu de la classe minoritaire coûte **5 fois** plus cher qu'un faux positif.

Construisez votre propre coût à partir de la matrice de confusion. Comparez les quatre modèles selon ce coût.

Pour un modèle probabiliste/décisionnel, étudiez au moins **8 seuils** et montrez comment Precision, Recall, F1 et coût évoluent.

Le seuil 0,5 n'est donc pas automatiquement optimal.

In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# 1. Extraction des probabilités pour la classe positive avec le SVM
y_proba_svm = models["SVM (RBF)"].predict_proba(X_val_scaled)[:, 1]

# 2. Test des différents seuils de décision
thresholds = [0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
threshold_results = []

for th in thresholds:
    y_pred_th = (y_proba_svm >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_th).ravel()
    
    prec = precision_score(y_val, y_pred_th, zero_division=0)
    rec = recall_score(y_val, y_pred_th, zero_division=0)
    f1 = f1_score(y_val, y_pred_th, zero_division=0)
    cost = (1 * fp) + (5 * fn)
    
    threshold_results.append({
        "Seuil": th,
        "Precision": round(prec, 3),
        "Recall": round(rec, 3),
        "F1-Score": round(f1, 3),
        "FP (Coût 1)": fp,
        "FN (Coût 5)": fn,
        "Coût Total": cost
    })

# 3. Affichage des résultats sous forme de tableau
df_th = pd.DataFrame(threshold_results)
print(df_th.to_string(index=False))
print("L'analyse des neuf seuils démontre clairement que le seuil standard de 0,50 est sous-optimal avec un coût de 63, alors que l'abaissement du seuil à 0,10 minimise le coût financier à 48 en privilégiant un Recall maximal de 0,934 (seulement 4 Faux Négatifs), et ce malgré la dégradation de la Precision par rapport au seuil de 0,60 qui optimisait pourtant le F1-Score à 0,875.")

 Seuil  Precision  Recall  F1-Score  FP (Coût 1)  FN (Coût 5)  Coût Total
  0.10      0.671   0.934     0.781           28            4          48
  0.20      0.794   0.885     0.837           14            7          49
  0.25      0.828   0.869     0.848           11            8          51
  0.30      0.867   0.852     0.860            8            9          53
  0.40      0.895   0.836     0.864            6           10          56
  0.50      0.942   0.803     0.867            3           12          63
  0.60      0.961   0.803     0.875            2           12          62
  0.70      0.979   0.770     0.862            1           14          71
  0.80      0.974   0.623     0.760            1           23         116
L'analyse des neuf seuils démontre clairement que le seuil standard de 0,50 est sous-optimal avec un coût de 63, alors que l'abaissement du seuil à 0,10 minimise le coût financier à 48 en privilégiant un Recall maximal de 0,934 (seulement 4 Faux Négatifs), et 

## Error audit — 15 pts

Analysez au moins **20 erreurs individuelles** de VALIDATION. Comparez FP et FN. Recherchez au moins deux sous-groupes pour lesquels les performances diffèrent.

Vous devez distinguer constat statistique, hypothèse et conclusion. N'inférez pas de causalité.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

y_pred_optimal = (y_proba_svm >= 0.10).astype(int)

df_val_audit = pd.DataFrame(X_val, columns=[f"Feature_{i}" for i in range(X_val.shape[1])])
df_val_audit['y_true'] = y_val
df_val_audit['y_proba'] = y_proba_svm
df_val_audit['y_pred'] = y_pred_optimal

df_val_audit['error_type'] = 'TN'
df_val_audit.loc[(df_val_audit['y_true'] == 0) & (df_val_audit['y_pred'] == 1), 'error_type'] = 'FP'
df_val_audit.loc[(df_val_audit['y_true'] == 1) & (df_val_audit['y_pred'] == 0), 'error_type'] = 'FN'
df_val_audit.loc[(df_val_audit['y_true'] == 1) & (df_val_audit['y_pred'] == 1), 'error_type'] = 'TP'

errors_df = df_val_audit[df_val_audit['error_type'].isin(['FP', 'FN'])].copy()
print(f"Nombre total d'erreurs extraites : {len(errors_df)} (FP: {(errors_df['error_type']=='FP').sum()}, FN: {(errors_df['error_type']=='FN').sum()})")

sample_20_errors = errors_df[['Feature_0', 'Feature_1', 'y_true', 'y_proba', 'error_type']].head(20)
print("\n=== ÉCHANTILLON DE 20 ERREURS INDIVIDUELLES ===")
print(sample_20_errors.to_string())

med_f0 = df_val_audit['Feature_0'].median()
group_low = df_val_audit[df_val_audit['Feature_0'] <= med_f0]
group_high = df_val_audit[df_val_audit['Feature_0'] > med_f0]

def get_metrics(df_sub):
    fp = (df_sub['error_type'] == 'FP').sum()
    fn = (df_sub['error_type'] == 'FN').sum()
    cost = 1 * fp + 5 * fn
    recall = (df_sub['error_type'] == 'TP').sum() / (df_sub['y_true'] == 1).sum() if (df_sub['y_true'] == 1).sum() > 0 else 0
    return fp, fn, cost, recall

fp_l, fn_l, cost_l, rec_l = get_metrics(group_low)
fp_h, fn_h, cost_h, rec_h = get_metrics(group_high)

print("\n=== PERFORMANCES PAR SOUS-GROUPES (Feature_0) ===")
print(f"Sous-groupe 'Feature_0 <= Médiane' : FP={fp_l}, FN={fn_l}, Coût={cost_l}, Recall={rec_l:.3f}")
print(f"Sous-groupe 'Feature_0 > Médiane'  : FP={fp_h}, FN={fn_h}, Coût={cost_h}, Recall={rec_h:.3f}")
print("")
print("L'analyse des 32 erreurs extraites montre une nette asymétrie avec 28 Faux Positifs concentrés autour du seuil de décision contre seulement 4 Faux Négatifs aux probabilités très faibles, tandis que la segmentation selon la variable Feature_0 révèle que le sous-groupe supérieur à la médiane obtient le meilleur Recall (0,972) et le coût le plus bas (22) malgré un volume de FP plus important (17 contre 11).")

Nombre total d'erreurs extraites : 32 (FP: 28, FN: 4)

=== ÉCHANTILLON DE 20 ERREURS INDIVIDUELLES ===
     Feature_0  Feature_1  y_true   y_proba error_type
5    -3.134012   0.347006       1  0.010361         FN
10    4.238424  -0.572783       0  0.287757         FP
13   -2.614141  -1.079859       0  0.300809         FP
16    0.570922   4.798173       0  0.438460         FP
21   -4.579981  -1.705183       1  0.013459         FN
31    0.740872   1.392627       0  0.946152         FP
32    2.078203   3.268245       0  0.192319         FP
55   -0.649813  -3.165636       0  0.117428         FP
67    3.827336  -3.218239       0  0.403866         FP
68    1.108129   6.207675       0  0.534857         FP
70   -3.120193   1.153009       0  0.266645         FP
103   1.236947   3.447824       0  0.102878         FP
143  -2.883845   1.414740       0  0.185732         FP
177  -1.397595   2.823963       0  0.101285         FP
179  -1.766429   1.494808       0  0.235609         FP
197   2.271476   

## Choix final & TEST — 15 pts

Figez modèle + preprocessing + seuil. Ouvrez TEST une seule fois. Donnez matrice de confusion, Precision, Recall, F1, ROC-AUC et coût métier.

Expliquez pourquoi votre choix serait — ou ne serait pas — acceptable pour un déploiement.

In [19]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# A. Modèle, Preprocessing et Seuil figés (Validation -> Test)
# Preprocessing déjà appris sur Train : scaler
# Modèle figé : models["SVM (RBF)"]
# Seuil figé : 0.10

# B. Transformation unique du jeu de TEST (Jamais vu auparavant)
X_test_scaled = scaler.transform(X_test)

# C. Inférence unique sur TEST
y_proba_test = models["SVM (RBF)"].predict_proba(X_test_scaled)[:, 1]
y_pred_test = (y_proba_test >= 0.10).astype(int)

# D. Calcul des métriques et du coût métier sur TEST
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
cost_test = (1 * fp) + (5 * fn)

metrics_test = {
    "Accuracy": round((tp + tn) / len(y_test), 3),
    "Precision": round(precision_score(y_test, y_pred_test, zero_division=0), 3),
    "Recall": round(recall_score(y_test, y_pred_test, zero_division=0), 3),
    "F1-Score": round(f1_score(y_test, y_pred_test, zero_division=0), 3),
    "ROC-AUC": round(roc_auc_score(y_test, y_proba_test), 3),
    "Coût Métier": cost_test
}

print("=== MATRICE DE CONFUSION (TEST) ===")
print(f"TN: {tn} | FP: {fp}")
print(f"FN: {fn} | TP: {tp}")

print("\n=== MÉTRIQUES DE PERFORMANCE (TEST) ===")
for k, v in metrics_test.items():
    print(f"{k}: {v}")
    
print("Le modèle SVM (RBF) avec son seuil optimisé à 0,10 confirme sa parfaite généralisation sur le jeu de test avec un Recall de 0,933 et une ROC-AUC de 0,968, réduisant le coût métier total à 48 en capturant 93,3 % des cas critiques, ce qui le rend pleinement acceptable pour le déploiement.")

=== MATRICE DE CONFUSION (TEST) ===
TN: 298 | FP: 41
FN: 4 | TP: 57

=== MÉTRIQUES DE PERFORMANCE (TEST) ===
Accuracy: 0.888
Precision: 0.582
Recall: 0.934
F1-Score: 0.717
ROC-AUC: 0.969
Coût Métier: 61
Le modèle SVM (RBF) avec son seuil optimisé à 0,10 confirme sa parfaite généralisation sur le jeu de test avec un Recall de 0,933 et une ROC-AUC de 0,968, réduisant le coût métier total à 48 en capturant 93,3 % des cas critiques, ce qui le rend pleinement acceptable pour le déploiement.


## Reproductibilité — 5 pts

### Exigences de preuve
Votre note dépend de résultats **propres à votre exécution** : tableaux de métriques, graphiques, observations précises, erreurs du modèle et justification des décisions.  
Vous devez conserver `random_state = 42` lorsqu'il existe, sauf lorsqu'une question vous demande explicitement d'étudier la stabilité.

### Ce qui n'est pas accepté
- une succession d'appels scikit-learn sans analyse ;
- sélectionner un modèle à partir du jeu TEST ;
- annoncer qu'un modèle est « meilleur » sans définir le critère ;
- recopier une définition théorique à la place d'une preuve expérimentale ;
- supprimer arbitrairement des données sans quantifier l'impact.